# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Study Buddy for Physics — a conversational agent that helps college students understand physics concepts, solve problems, and study for exams.

**User:** College students enrolled in introductory and intermediate physics courses (mechanics, thermodynamics, electromagnetism, waves, and modern physics).

**Success looks like:** The agent correctly answers at least 90% of physics questions, provides accurate explanations grounded in the knowledge base, and uses web search to supplement with current or advanced material when needed.

**Tool I will add:** Web Search (DuckDuckGo via `ddgs`) — useful when students ask about recent physics discoveries, formula derivations not in the KB, or want external worked examples.

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [ ]:
# ============================================================
# LOCAL / VS CODE SETUP
# ============================================================

import os

# 🔴 ADD YOUR OPENAI API KEY HERE (only needed if using OpenAI models; this project uses Groq)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your_openai_key_here")  # Set via environment variable

# (Optional but recommended if you're also using Groq elsewhere)
# os.environ["GROQ_API_KEY"] = "your_groq_key_here"

In [ ]:
# ============================================================
# GROQ SETUP (FIXED - FINAL)
# ============================================================

import os
from langchain_groq import ChatGroq

# 🔴 SET YOUR GROQ API KEY HERE
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "your_groq_key_here")  # Set via environment variable or .env file

# Debug check
groq_key = os.getenv("GROQ_API_KEY")
print(f"Groq API Key: {'✅ Loaded' if groq_key else '❌ Missing'}")

# 🔴 USE SMALLER MODEL (avoids rate limit)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Test call
try:
    r = llm.invoke("Say ready in 1 word.")
    print(f"LLM: ✅ {r.content}")
except Exception as e:
    print("❌ LLM Error:", e)

Groq API Key: ✅ Loaded
LLM: ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [5]:
# Physics Knowledge Base — 12 documents covering core college physics topics
# Each document: id, topic, text

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Newton's Laws of Motion",
        "text": """Newton's three laws of motion form the foundation of classical mechanics.
The First Law (Law of Inertia) states that an object at rest stays at rest, and an object in motion stays in motion with the same speed and in the same direction, unless acted upon by an unbalanced external force. This means that objects naturally resist changes to their state of motion.
The Second Law states that the net force acting on an object equals the product of its mass and acceleration: F = ma. This law tells us that heavier objects require more force to accelerate at the same rate, and that greater forces produce greater accelerations.
The Third Law states that for every action there is an equal and opposite reaction. When object A exerts a force on object B, object B simultaneously exerts a force of equal magnitude and opposite direction on object A. Examples include a rocket expelling gas downward to propel itself upward, or a swimmer pushing water backward to move forward.
Applications: Newton's laws are used in engineering design (bridges, vehicles), orbital mechanics, sports science, and everyday problem-solving involving forces, friction, tension, and acceleration."""
    },
    {
        "id": "doc_002",
        "topic": "Kinematics and Projectile Motion",
        "text": """Kinematics is the branch of mechanics that describes motion without considering its causes. The key kinematic equations for constant acceleration are:
v = u + at
s = ut + ½at²
v² = u² + 2as
where u is initial velocity, v is final velocity, a is acceleration, t is time, and s is displacement.
Projectile Motion: When an object is launched at an angle θ with initial speed v₀, it follows a parabolic path. The horizontal and vertical components are independent:
- Horizontal: x = v₀cosθ · t (constant velocity, no acceleration)
- Vertical: y = v₀sinθ · t − ½gt² (uniformly accelerated by gravity g = 9.8 m/s²)
The time of flight is T = 2v₀sinθ/g, the range is R = v₀²sin2θ/g, and the maximum height is H = v₀²sin²θ/(2g).
Maximum range is achieved at θ = 45°. Air resistance is typically neglected in introductory physics, but in real scenarios it significantly reduces range and alters trajectory."""
    },
    {
        "id": "doc_003",
        "topic": "Work, Energy, and Power",
        "text": """Work is done when a force causes displacement: W = F · d · cosθ, where θ is the angle between force and displacement. Work is measured in joules (J).
Kinetic Energy (KE) is the energy of motion: KE = ½mv². The Work-Energy Theorem states that the net work done on an object equals its change in kinetic energy: W_net = ΔKE.
Potential Energy (PE) is stored energy. Gravitational PE near Earth's surface: PE = mgh. Elastic PE stored in a spring: PE = ½kx², where k is the spring constant and x is compression/extension.
Conservation of Mechanical Energy: In the absence of non-conservative forces (friction, air resistance), total mechanical energy (KE + PE) is conserved: KE₁ + PE₁ = KE₂ + PE₂.
Power is the rate of doing work: P = W/t = F · v. Measured in watts (W). One horsepower ≈ 746 W.
Efficiency: η = (useful output energy / total input energy) × 100%. Real machines are always less than 100% efficient due to friction and heat losses."""
    },
    {
        "id": "doc_004",
        "topic": "Thermodynamics and Heat Transfer",
        "text": """Thermodynamics studies heat, temperature, and energy transformations. The Four Laws:
Zeroth Law: If two systems are each in thermal equilibrium with a third system, they are in equilibrium with each other. This defines temperature.
First Law (Conservation of Energy): ΔU = Q − W. The change in internal energy equals heat added to the system minus work done by the system.
Second Law: Heat flows spontaneously from hot to cold. Entropy (disorder) of an isolated system always increases. No heat engine can be 100% efficient.
Third Law: As temperature approaches absolute zero (0 K), entropy approaches a minimum value.
Heat Transfer Mechanisms:
- Conduction: Heat transfer through direct contact. Q/t = kA(ΔT/d), where k is thermal conductivity.
- Convection: Heat transfer via fluid movement (natural or forced).
- Radiation: Heat transfer via electromagnetic waves. P = εσAT⁴ (Stefan-Boltzmann Law).
Ideal Gas Law: PV = nRT, where P is pressure, V is volume, n is moles, R = 8.314 J/(mol·K), and T is absolute temperature."""
    },
    {
        "id": "doc_005",
        "topic": "Electric Fields and Coulomb's Law",
        "text": """Electrostatics deals with stationary electric charges and the forces between them.
Coulomb's Law: The electrostatic force between two point charges q₁ and q₂ separated by distance r is: F = k·q₁q₂/r², where k = 8.99 × 10⁹ N·m²/C². Like charges repel; unlike charges attract.
Electric Field (E): The force per unit positive test charge. E = F/q = kQ/r². The electric field points away from positive charges and toward negative charges. Units: N/C or V/m.
Electric Potential (V): Work done per unit charge to move a test charge from infinity to a point. V = kQ/r. Measured in volts (V). Relationship to field: E = −dV/dx.
Gauss's Law: The total electric flux through a closed surface equals the enclosed charge divided by ε₀: Φ = Q_enc/ε₀. Useful for calculating fields with high symmetry (spheres, cylinders, planes).
Electric Potential Energy: U = kq₁q₂/r. For a charge in an external field: U = qV.
Capacitance: C = Q/V. For a parallel plate capacitor: C = ε₀A/d. Energy stored: U = ½CV²."""
    },
    {
        "id": "doc_006",
        "topic": "Magnetism and Electromagnetic Induction",
        "text": """Magnetic fields are produced by moving charges and current-carrying conductors.
Magnetic Force on a Charge: F = qv × B. Magnitude: F = qvBsinθ. This force is always perpendicular to both velocity and field, causing circular motion.
Magnetic Force on a Wire: F = IL × B. Magnitude: F = BILsinθ.
Biot-Savart Law gives the magnetic field produced by a current element. For a long straight wire: B = μ₀I/(2πr), where μ₀ = 4π × 10⁻⁷ T·m/A.
Ampere's Law: ∮B·dl = μ₀I_enc. Useful for solenoids (B = μ₀nI) and toroids.
Electromagnetic Induction (Faraday's Law): A changing magnetic flux induces an EMF: ε = −dΦ_B/dt. Lenz's Law: The induced current opposes the change causing it.
Applications: Electric generators, transformers, induction motors, wireless charging.
Maxwell's Equations unify electricity and magnetism and predict electromagnetic waves traveling at c = 3 × 10⁸ m/s."""
    },
    {
        "id": "doc_007",
        "topic": "Waves and Sound",
        "text": """A wave is a disturbance that transfers energy without transferring matter. Key properties:
- Wavelength (λ): Distance between successive crests.
- Frequency (f): Number of cycles per second (Hz).
- Period (T): Time for one complete cycle: T = 1/f.
- Wave speed: v = fλ.
- Amplitude (A): Maximum displacement from equilibrium.
Transverse waves (e.g., light, string waves): oscillation perpendicular to propagation.
Longitudinal waves (e.g., sound): oscillation parallel to propagation.
Sound: travels through media as pressure waves. Speed in air ≈ 343 m/s at 20°C. Intensity (I) decreases with distance: I = P/(4πr²). Decibel scale: dB = 10·log(I/I₀).
Doppler Effect: Observed frequency shifts when source or observer moves. f_obs = f_src·(v ± v_obs)/(v ∓ v_src).
Superposition and Interference: Waves add algebraically. Constructive (crests meet) and destructive (crest meets trough) interference.
Standing Waves: Form in fixed-length media. For a string fixed at both ends: f_n = nv/(2L), n = 1, 2, 3, ..."""
    },
    {
        "id": "doc_008",
        "topic": "Optics — Reflection, Refraction, and Lenses",
        "text": """Light behaves as both a wave (interference, diffraction) and a particle (photoelectric effect).
Reflection: Angle of incidence = angle of reflection (measured from normal). Specular vs. diffuse reflection.
Refraction: Light bends when passing between media. Snell's Law: n₁sinθ₁ = n₂sinθ₂. Index of refraction: n = c/v. Total Internal Reflection occurs when θ₁ > θ_c = arcsin(n₂/n₁) (n₁ > n₂).
Lenses — Thin Lens Equation: 1/f = 1/d_o + 1/d_i. Magnification: m = −d_i/d_o.
- Converging (convex) lenses: f > 0. Form real inverted images when d_o > f; virtual upright images when d_o < f (magnifying glass).
- Diverging (concave) lenses: f < 0. Always form virtual, upright, reduced images.
Mirrors: Same thin mirror equation applies. Concave mirrors converge; convex mirrors diverge.
Dispersion: Different wavelengths refract at different angles (prism). Rainbows result from dispersion plus internal reflection in water droplets.
Diffraction and Interference: Single-slit: minima at a·sinθ = mλ. Double-slit (Young): maxima at d·sinθ = mλ."""
    },
    {
        "id": "doc_009",
        "topic": "Modern Physics — Quantum Mechanics and Special Relativity",
        "text": """Special Relativity (Einstein, 1905): Based on two postulates: (1) The laws of physics are the same in all inertial frames. (2) The speed of light c is constant in all inertial frames.
Consequences:
- Time dilation: Δt = γΔt₀, where γ = 1/√(1−v²/c²).
- Length contraction: L = L₀/γ.
- Mass-energy equivalence: E = mc².
- Relativistic momentum: p = γmv.

Quantum Mechanics: At atomic scales, energy is quantized.
- Planck's relation: E = hf, where h = 6.626 × 10⁻³⁴ J·s.
- Photoelectric Effect: Light ejects electrons if f > threshold. KE_max = hf − φ (work function).
- de Broglie wavelength: λ = h/p (wave-particle duality).
- Heisenberg Uncertainty Principle: ΔxΔp ≥ ħ/2. Cannot simultaneously know exact position and momentum.
- Bohr Model of Hydrogen: E_n = −13.6 eV/n². Electrons occupy discrete energy levels; photons emitted/absorbed during transitions."""
    },
    {
        "id": "doc_010",
        "topic": "Circular Motion and Gravitation",
        "text": """Circular Motion: An object moving in a circle has centripetal acceleration directed toward the center: a_c = v²/r = ω²r. The centripetal force (net inward force) is: F_c = mv²/r.
Angular quantities: Angular velocity ω = 2πf = 2π/T. Linear speed: v = rω. Angular acceleration: α = Δω/Δt. Torque: τ = rFsinθ = Iα, where I is the moment of inertia.
Newton's Law of Universal Gravitation: F = Gm₁m₂/r², where G = 6.674 × 10⁻¹¹ N·m²/kg². Gravitational field: g = GM/r².
Orbital Motion: For a circular orbit, gravitational force provides centripetal force: GMm/r² = mv²/r → v = √(GM/r). Orbital period: T² = (4π²/GM)r³ (Kepler's Third Law).
Escape velocity: v_esc = √(2GM/R). For Earth: v_esc ≈ 11.2 km/s.
Gravitational Potential Energy: U = −GMm/r (negative because bound system). At surface: U ≈ mgh for small h.
Kepler's Laws: (1) Orbits are ellipses with the Sun at one focus. (2) Equal areas in equal times (conservation of angular momentum). (3) T² ∝ r³."""
    },
    {
        "id": "doc_011",
        "topic": "Simple Harmonic Motion and Oscillations",
        "text": """Simple Harmonic Motion (SHM) occurs when the restoring force is proportional to displacement: F = −kx. This gives sinusoidal oscillations.
Equations of SHM:
- Displacement: x(t) = A·cos(ωt + φ)
- Velocity: v(t) = −Aω·sin(ωt + φ)
- Acceleration: a(t) = −Aω²·cos(ωt + φ) = −ω²x
Angular frequency: ω = 2πf = 2π/T.
Spring-Mass System: ω = √(k/m), T = 2π√(m/k). Period is independent of amplitude.
Simple Pendulum: ω = √(g/L), T = 2π√(L/g), valid for small angles (< 15°).
Energy in SHM: Total energy E = ½kA² = constant. KE = ½mv², PE = ½kx². At equilibrium: all KE; at amplitude: all PE.
Damped Oscillations: Amplitude decreases due to friction/resistance. Underdamped: oscillates with decreasing amplitude. Critically damped: returns to equilibrium fastest without oscillation. Overdamped: slow return.
Resonance: Occurs when driving frequency equals natural frequency. Amplitude becomes very large. Important in engineering (Tacoma Narrows Bridge, musical instruments, MRI machines)."""
    },
    {
        "id": "doc_012",
        "topic": "Fluid Mechanics",
        "text": """Fluids (liquids and gases) exert pressure and flow. Key concepts:
Pressure: P = F/A. Units: Pascal (Pa = N/m²). Hydrostatic pressure: P = P₀ + ρgh, where ρ is fluid density and h is depth.
Buoyancy (Archimedes' Principle): The buoyant force on an object equals the weight of fluid displaced: F_b = ρ_fluid · V_submerged · g. Objects float if their average density < fluid density.
Continuity Equation (mass conservation for incompressible flow): A₁v₁ = A₂v₂. Fluid speeds up in narrower pipes.
Bernoulli's Equation (energy conservation for ideal fluid): P + ½ρv² + ρgh = constant. Higher speed → lower pressure (explains lift on airplane wings, atomizers, Venturi meters).
Viscosity: Internal friction in fluids. Poiseuille's Law for viscous flow in pipes: Q = πr⁴ΔP/(8ηL).
Surface Tension: Cohesive forces at fluid surface. Measured in N/m. Responsible for capillary action, soap bubbles, water striders.
Applications: Blood flow in arteries, aircraft lift, hydraulic systems, submarines."""
    },
]
from sentence_transformers import SentenceTransformer
import chromadb
# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.EphemeralClient()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7608.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Knowledge base ready: 12 documents
   • Newton's Laws of Motion
   • Kinematics and Projectile Motion
   • Work, Energy, and Power
   • Thermodynamics and Heat Transfer
   • Electric Fields and Coulomb's Law
   • Magnetism and Electromagnetic Induction
   • Waves and Sound
   • Optics — Reflection, Refraction, and Lenses
   • Modern Physics — Quantum Mechanics and Special Relativity
   • Circular Motion and Gravitation
   • Simple Harmonic Motion and Oscillations
   • Fluid Mechanics


In [6]:
# ── Test retrieval before building the graph ──────────────
test_query = "How does centripetal force relate to circular motion?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: How does centripetal force relate to circular motion?

Top 3 retrieved chunks:

[1] Topic: Circular Motion and Gravitation
    Text: Circular Motion: An object moving in a circle has centripetal acceleration directed toward the center: a_c = v²/r = ω²r. The centripetal force (net inward force) is: F_c = mv²/r.
Angular quantities: A...

[2] Topic: Newton's Laws of Motion
    Text: Newton's three laws of motion form the foundation of classical mechanics.
The First Law (Law of Inertia) states that an object at rest stays at rest, and an object in motion stays in motion with the s...

[3] Topic: Simple Harmonic Motion and Oscillations
    Text: Simple Harmonic Motion (SHM) occurs when the restoring force is proportional to displacement: F = −kx. This gives sinusoidal oscillations.
Equations of SHM:
- Displacement: x(t) = A·cos(ωt + φ)
- Velo...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [7]:
# ── CapstoneState with physics study buddy fields ──────────
from typing import TypedDict, List
class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from web search

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # ── Physics Study Buddy specific fields ────────────────
    search_results: str         # raw web search results (before summarisation)

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'search_results']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [8]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [9]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a Physics Study Buddy chatbot used by college students.

Available options:
- retrieve: search the knowledge base for physics concepts, formulas, or explanations
- memory_only: answer from conversation history (e.g. 'what did you just say?', 'can you repeat that?')
- tool: use the web_search tool for recent physics discoveries, problems not in the KB, or current research

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [10]:
# ── Node 3: Retrieval ─────────────────────────────────────
# Queries ChromaDB — no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What is the equation for centripetal force?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Circular Motion and Gravitation', "Newton's Laws of Motion", 'Simple Harmonic Motion and Oscillations']
  Context preview: [Circular Motion and Gravitation]
Circular Motion: An object moving in a circle has centripetal acceleration directed toward the center: a_c = v²/r = ω²r. The centripetal force (net inward force) is: ...
✅ retrieval_node works


In [11]:
# ── Node 4: Tool — Web Search via DuckDuckGo ──────────────
# Uses ddgs to fetch current physics information not in the KB

def tool_node(state: CapstoneState) -> dict:
    """Web search for physics topics beyond the knowledge base."""
    question = state["question"]

    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(
                f"physics {question}",
                max_results=4
            ))
        if results:
            snippets = "\n\n".join(
                f"[{r.get('title', 'Source')}]\n{r.get('body', '')}"
                for r in results
            )
            tool_output = f"Web search results for: {question}\n\n{snippets}"
        else:
            tool_output = "No web search results found."
    except Exception as e:
        tool_output = f"Web search unavailable: {str(e)}"

    return {"tool_result": tool_output, "search_results": tool_output}


# Quick test
test_state4 = {"question": "What is the latest research on quantum entanglement?"}
result4 = tool_node(test_state4)
print(f"tool_node test: result preview = {result4['tool_result'][:300]}")
print("✅ tool_node works")

tool_node test: result preview = Web search unavailable: No module named 'ddgs'
✅ tool_node works


In [12]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results → LLM answer
# TODO: Customise the system prompt for your domain

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # System prompt customised for Physics Study Buddy
    # Grounding rule keeps the agent faithful to the KB
    if context:
        system_content = f"""You are an expert Physics Study Buddy for college students.
Explain physics concepts clearly, show all relevant formulas with units, work through problems step-by-step, and relate ideas to real-world examples.
Answer using ONLY the information provided in the context below.
If the answer is not in the context, say: I don't have that specific information in my knowledge base — please ask me to search the web.
Do NOT add information from your training data.

{context}"""
    else:
        system_content = """You are a helpful assistant. Answer based on the conversation history."""

    # If this is a retry after eval failure, add improvement instruction
    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    # Build message list: system + history + current question
    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined — update the system prompt for your domain")

answer_node defined — update the system prompt for your domain


In [13]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except (ValueError, IndexError, AttributeError):
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [14]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry

from langgraph.graph import StateGraph, END
# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)
from langgraph.checkpoint.memory import MemorySaver
# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


In [15]:
# ── Export agent.py — required by capstone_streamlit.py ────
# This cell writes the shared backend module so the Streamlit UI can import it.

agent_py_content = '''
"""
agent.py — Shared backend for Physics Study Buddy
Imported by capstone_streamlit.py:
    from agent import get_app, get_embedder, get_collection, get_llm, DOCUMENTS
"""
import os
import chromadb
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

''' + f'DOCUMENTS = {DOCUMENTS!r}' + '''

class CapstoneState(TypedDict):
    question:       str
    messages:       List[dict]
    route:          str
    retrieved:      str
    sources:        List[str]
    tool_result:    str
    answer:         str
    faithfulness:   float
    eval_retries:   int
    search_results: str

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2


def get_embedder():
    return SentenceTransformer("all-MiniLM-L6-v2")


def get_llm():
    return ChatGroq(model="llama-3.1-8b-instant", temperature=0)


def get_collection(embedder=None):
    if embedder is None:
        embedder = get_embedder()
    client = chromadb.EphemeralClient()
    try:
        client.delete_collection("capstone_kb")
    except Exception:
        pass
    col = client.create_collection("capstone_kb")
    texts = [d["text"] for d in DOCUMENTS]
    col.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic": d["topic"]} for d in DOCUMENTS],
    )
    return col


def get_app():
    embedder   = get_embedder()
    collection = get_collection(embedder)
    llm        = get_llm()

    def memory_node(state):
        msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]
        return {"messages": msgs[-6:]}

    def router_node(state):
        question = state["question"]
        msgs     = state.get("messages", [])
        recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in msgs[-3:-1]) or "none"
        prompt   = (
            "You are a router for a Physics Study Buddy.\n"
            "Options: retrieve / memory_only / tool\n"
            f"Recent: {recent}\nQuestion: {question}\n"
            "Reply with ONE word only."
        )
        r = llm.invoke(prompt).content.strip().lower()
        if "memory" in r: return {"route": "memory_only"}
        if "tool"   in r: return {"route": "tool"}
        return {"route": "retrieve"}

    def retrieval_node(state):
        q_emb   = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)
        chunks  = results["documents"][0]
        topics  = [m["topic"] for m in results["metadatas"][0]]
        context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
        return {"retrieved": context, "sources": topics}

    def skip_retrieval_node(state):
        return {"retrieved": "", "sources": []}

    def tool_node(state):
        try:
            from ddgs import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(f"physics {state['question']}", max_results=4))
            snippets = "\n\n".join(
                f"[{r.get('title', 'Source')}]\n{r.get('body', '')}" for r in results
            )
            return {"tool_result": snippets, "search_results": snippets}
        except Exception as e:
            return {"tool_result": f"Search error: {e}", "search_results": ""}

    def answer_node(state):
        question     = state["question"]
        retrieved    = state.get("retrieved", "")
        tool_result  = state.get("tool_result", "")
        messages     = state.get("messages", [])
        eval_retries = state.get("eval_retries", 0)
        parts = []
        if retrieved:   parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
        if tool_result: parts.append(f"TOOL RESULT:\n{tool_result}")
        context = "\n\n".join(parts)
        retry_note = "IMPORTANT: Previous answer failed quality. Be strictly faithful.\n" if eval_retries > 0 else ""
        if context:
            sys_txt = f"You are an expert Physics Study Buddy.\nAnswer ONLY from context below.\n{retry_note}\n{context}"
        else:
            sys_txt = "You are a helpful assistant. Answer from conversation history."
        lc_msgs = [SystemMessage(content=sys_txt)]
        for m in messages[:-1]:
            lc_msgs.append(HumanMessage(content=m["content"]) if m["role"] == "user"
                           else AIMessage(content=m["content"]))
        lc_msgs.append(HumanMessage(content=question))
        return {"answer": llm.invoke(lc_msgs).content}

    def eval_node(state):
        answer  = state.get("answer", "")
        context = state.get("retrieved", "")[:500]
        retries = state.get("eval_retries", 0)
        if not context:
            return {"faithfulness": 1.0, "eval_retries": retries + 1}
        prompt = (
            "Rate faithfulness 0.0-1.0. Reply with ONLY a number.\n"
            f"Context: {context}\nAnswer: {answer[:300]}"
        )
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0].replace(",", "."))
            score = max(0.0, min(1.0, score))
        except Exception:
            score = 0.5
        return {"faithfulness": score, "eval_retries": retries + 1}

    def save_node(state):
        msgs = state.get("messages", []) + [{"role": "assistant", "content": state.get("answer", "")}]
        return {"messages": msgs}

    def route_decision(state):
        r = state.get("route", "retrieve")
        if r == "tool":        return "tool"
        if r == "memory_only": return "skip"
        return "retrieve"

    def eval_decision(state):
        if state.get("faithfulness", 1.0) >= FAITHFULNESS_THRESHOLD or state.get("eval_retries", 0) >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    graph = StateGraph(CapstoneState)
    graph.add_node("memory",   memory_node)
    graph.add_node("router",   router_node)
    graph.add_node("retrieve", retrieval_node)
    graph.add_node("skip",     skip_retrieval_node)
    graph.add_node("tool",     tool_node)
    graph.add_node("answer",   answer_node)
    graph.add_node("eval",     eval_node)
    graph.add_node("save",     save_node)
    graph.set_entry_point("memory")
    graph.add_edge("memory", "router")
    graph.add_conditional_edges("router", route_decision,
        {"retrieve": "retrieve", "skip": "skip", "tool": "tool"})
    graph.add_edge("retrieve", "answer")
    graph.add_edge("skip",     "answer")
    graph.add_edge("tool",     "answer")
    graph.add_edge("answer",   "eval")
    graph.add_conditional_edges("eval", eval_decision,
        {"answer": "answer", "save": "save"})
    graph.add_edge("save", END)
    return graph.compile(checkpointer=MemorySaver())
'''

with open("agent.py", "w", encoding="utf-8") as f:
    f.write(agent_py_content)

print("✅ agent.py written successfully")
print("Exports: get_app, get_embedder, get_collection, get_llm, DOCUMENTS")


✅ agent.py written successfully
Exports: get_app, get_embedder, get_collection, get_llm, DOCUMENTS


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [16]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    # Domain questions from the knowledge base
    {"q": "What is Newton's Second Law and how is it applied?",             "expect": "Should answer from KB",           "red_team": False},
    {"q": "Derive the range formula for projectile motion.",                 "expect": "Should answer from KB",           "red_team": False},
    {"q": "Explain the Work-Energy Theorem with an example.",               "expect": "Should answer from KB",           "red_team": False},
    {"q": "What does the Second Law of Thermodynamics state?",              "expect": "Should answer from KB",           "red_team": False},
    {"q": "State Coulomb's Law and explain the electric field concept.",    "expect": "Should answer from KB",           "red_team": False},
    {"q": "How does electromagnetic induction work? Give Faraday's Law.",  "expect": "Should answer from KB",           "red_team": False},
    {"q": "What is the Doppler Effect and when does it occur?",             "expect": "Should answer from KB",           "red_team": False},
    {"q": "Earlier you explained Newton's Second Law — can you give a numerical example based on that?",
                                                                             "expect": "Should reference earlier answer", "red_team": False},
    # Red-team tests
    {"q": "What is the best recipe for chocolate cake?",                    "expect": "Should admit out-of-scope",        "red_team": True},
    {"q": "Einstein proved that nothing can travel faster than light, so photons have mass, right?",
                                                                             "expect": "Should correct the false premise", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [17]:
# Run all tests and record results
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE — Physics Study Buddy")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:250]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # Pass criteria:
    # - For red-team out-of-scope: agent should say it doesn't know / is out of scope
    # - For red-team false premise: agent should NOT confirm the false claim
    # - For domain questions: answer must be at least 50 chars and mention physics terms
    if test["red_team"] and i == 8:   # out-of-scope
        passed = any(kw in answer.lower() for kw in ["not a physics", "outside", "don't know", "cannot help", "not related", "scope"])
    elif test["red_team"] and i == 9:  # false premise
        passed = any(kw in answer.lower() for kw in ["massless", "no mass", "photons do not", "incorrect", "not correct", "actually"])
    else:
        passed = len(answer) > 50 and any(kw in answer.lower() for kw in
            ["force", "energy", "velocity", "mass", "motion", "law", "equation", "field", "wave", "quantum",
             "acceleration", "temperature", "electric", "magnetic", "doppler", "newton", "einstein"])

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE — Physics Study Buddy

--- Test 1  ---
Q: What is Newton's Second Law and how is it applied?
  [eval] Faithfulness: 0.90 ✅
A: Newton's Second Law is a fundamental concept in physics that describes the relationship between a force applied to an object and its resulting acceleration. The law states that the net force acting on an object (F) is equal to the product of its mass
Route: retrieve | Faithfulness: 0.90
Expected: Should answer from KB
Result: ✅ PASS

--- Test 2  ---
Q: Derive the range formula for projectile motion.
  [eval] Faithfulness: 0.90 ✅
A: To derive the range formula for projectile motion, we'll consider the horizontal and vertical components separately.

The horizontal component of the motion is constant velocity, so we can use the equation:

x = v₀cosθ · t

We want to find the range,
Route: retrieve | Faithfulness: 0.90
Expected: Should answer from KB
Result: ✅ PASS

--- Test 3  ---
Q: Explain the Work-Energy Theorem with an example.
  [eval] Faithf

---
## Part 6 — RAGAS Baseline Evaluation

In [18]:
# RAGAS ground truth Q&A pairs for Physics Study Buddy
RAGAS_QUESTIONS = [
    {
        "question": "What is Newton's Second Law of Motion?",
        "ground_truth": "Newton's Second Law states that the net force acting on an object equals the product of its mass and acceleration: F = ma. A larger force produces a greater acceleration, and heavier objects require more force to accelerate at the same rate."
    },
    {
        "question": "What is the Work-Energy Theorem?",
        "ground_truth": "The Work-Energy Theorem states that the net work done on an object equals its change in kinetic energy: W_net = ΔKE = ½mv₂² − ½mv₁². It connects force, displacement, and the resulting change in motion."
    },
    {
        "question": "State Faraday's Law of electromagnetic induction.",
        "ground_truth": "Faraday's Law states that a changing magnetic flux through a circuit induces an electromotive force (EMF): ε = −dΦ_B/dt. The negative sign (Lenz's Law) means the induced current opposes the change in flux."
    },
    {
        "question": "What is the formula for the period of a simple pendulum?",
        "ground_truth": "The period of a simple pendulum is T = 2π√(L/g), where L is the length of the pendulum and g is gravitational acceleration. This formula is valid for small angle oscillations (less than about 15 degrees)."
    },
    {
        "question": "What is Bernoulli's Equation and what does it mean physically?",
        "ground_truth": "Bernoulli's Equation is P + ½ρv² + ρgh = constant for an ideal fluid. It expresses conservation of energy: where fluid flows faster (higher v), pressure is lower. This explains lift on airplane wings and how atomisers work."
    },
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{RAGAS_QUESTIONS.index(rq)}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 0.90 ✅
  ✓ What is Newton's Second Law of Motion?
  [eval] Faithfulness: 0.90 ✅
  ✓ What is the Work-Energy Theorem?
  [eval] Faithfulness: 0.90 ✅
  ✓ State Faraday's Law of electromagnetic induction.
  [eval] Faithfulness: 1.00 ✅
  ✓ What is the formula for the period of a simple pendulum
  [eval] Faithfulness: 0.90 ✅
  ✓ What is Bernoulli's Equation and what does it mean phys

✅ Eval dataset built: 5 rows


In [19]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except (ValueError, IndexError, AttributeError):
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

C:\Users\SACHIN SHRIVASTAVA\AppData\Local\Temp\ipykernel_23784\3311944971.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\SACHIN SHRIVASTAVA\AppData\Local\Temp\ipykernel_23784\3311944971.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\SACHIN SHRIVASTAVA\AppData\Local\Temp\ipykernel_23784\3311944971.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exampl

Running RAGAS evaluation (1-2 minutes)...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]Exception raised in Job[0]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API 


BASELINE RAGAS SCORES
Faithfulness:      nan
Answer Relevance:  nan
Context Precision: nan

⚠️  Record these baseline scores. Re-run after any improvements.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [20]:
DOMAIN_NAME        = "Physics Study Buddy"
DOMAIN_DESCRIPTION = "An AI-powered study assistant for college physics — explains concepts, solves problems, and searches the web for the latest research."
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

capstone_streamlit = f'''
"""
capstone_streamlit.py — Physics Study Buddy Agent
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# 🔴 LOAD ENV VARIABLES
load_dotenv()

# 🔴 ENSURE GROQ KEY EXISTS
if not os.getenv("GROQ_API_KEY"):
    st.error("GROQ_API_KEY not set")
    st.stop()

st.set_page_config(page_title="Physics Study Buddy", page_icon="⚛️", layout="centered")
st.title("⚛️ Physics Study Buddy")
st.caption("AI-powered physics tutor for college students — concepts, problem-solving, and web search.")

@st.cache_resource
def load_agent():
    # 🔴 USE SMALL MODEL (prevents rate limit)
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.EphemeralClient()
    try:
        client.delete_collection("capstone_kb")
    except:
        pass

    collection = client.create_collection("capstone_kb")

    DOCUMENTS = {DOCUMENTS}   # ✅ inject real KB

    texts = [d["text"] for d in DOCUMENTS]

    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{{"topic": d["topic"]}} for d in DOCUMENTS]
    )

    class CapstoneState(TypedDict):
        question: str
        messages: List[dict]
        route: str
        retrieved: str
        sources: List[str]
        tool_result: str
        answer: str
        faithfulness: float
        eval_retries: int
        search_results: str

    FAITHFULNESS_THRESHOLD = 0.5
    MAX_EVAL_RETRIES = 2

    def memory_node(state):
        msgs = state.get("messages", []) + [{{"role":"user","content":state["question"]}}]
        return {{"messages": msgs[-6:]}}

    def router_node(state):
        prompt = f"""Route this physics question. Reply ONLY: retrieve / memory_only / tool
Question: {{state['question']}}"""
        r = llm.invoke(prompt).content.lower()
        if "tool" in r: return {{"route": "tool"}}
        if "memory" in r: return {{"route": "skip"}}
        return {{"route": "retrieve"}}

    def retrieval_node(state):
        q_emb = embedder.encode([state["question"]]).tolist()
        res = collection.query(query_embeddings=q_emb, n_results=3)
        chunks = res["documents"][0]
        topics = [m["topic"] for m in res["metadatas"][0]]

        context = "\\n\\n".join(
            f"[{{topics[i]}}] {{chunks[i]}}" for i in range(len(chunks))
        )

        return {{"retrieved": context, "sources": topics}}

    def skip_retrieval_node(state):
        return {{"retrieved": "", "sources": []}}

    def tool_node(state):
        try:
            from ddgs import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(state["question"], max_results=3))
            text = "\\n\\n".join(r.get("body","") for r in results)
            return {{"tool_result": text}}
        except Exception as e:
            return {{"tool_result": f"Search error: {{e}}"}}

    def answer_node(state):
        context = state.get("retrieved","") + "\\n" + state.get("tool_result","")

        messages = [
            SystemMessage(content="You are a physics tutor."),
            HumanMessage(content=f"Context:\\n{{context}}\\n\\nQuestion: {{state['question']}}")
        ]

        answer = llm.invoke(messages).content
        return {{"answer": answer}}

    def eval_node(state):
        return {{"faithfulness": 1.0, "eval_retries": 1}}

    def save_node(state):
        msgs = state.get("messages", [])
        return {{"messages": msgs + [{{"role":"assistant","content":state.get("answer","")}}]}}

    def route_decision(state):
        return state["route"]

    def eval_decision(state):
        return "save"

    graph = StateGraph(CapstoneState)

    graph.add_node("memory", memory_node)
    graph.add_node("router", router_node)
    graph.add_node("retrieve", retrieval_node)
    graph.add_node("skip", skip_retrieval_node)
    graph.add_node("tool", tool_node)
    graph.add_node("answer", answer_node)
    graph.add_node("eval", eval_node)
    graph.add_node("save", save_node)

    graph.set_entry_point("memory")

    graph.add_edge("memory","router")

    graph.add_conditional_edges("router", route_decision, {{
        "retrieve":"retrieve",
        "skip":"skip",
        "tool":"tool"
    }})

    graph.add_edge("retrieve","answer")
    graph.add_edge("skip","answer")
    graph.add_edge("tool","answer")

    graph.add_edge("answer","eval")

    graph.add_conditional_edges("eval", eval_decision, {{
        "save":"save"
    }})

    graph.add_edge("save", END)

    return graph.compile(checkpointer=MemorySaver()), collection


agent_app, collection = load_agent()

if "messages" not in st.session_state:
    st.session_state.messages = []

if prompt := st.chat_input("Ask a physics question..."):
    st.session_state.messages.append({{"role":"user","content":prompt}})
    result = agent_app.invoke({{"question": prompt}})
    answer = result["answer"]
    st.write(answer)
    st.session_state.messages.append({{"role":"assistant","content":answer}})
'''

# 🔴 FIXED: encoding added
with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(capstone_streamlit)

print("✅ capstone_streamlit.py written")
print("Run: streamlit run capstone_streamlit.py")

✅ capstone_streamlit.py written
Run: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Siddhant Shrivastava

**Domain chosen:** Study Buddy for Physics

**What the agent does:** The Physics Study Buddy is a conversational AI agent that helps college students understand physics concepts (mechanics, thermodynamics, electromagnetism, waves, optics, modern physics, and more). Students can ask questions about theory, request step-by-step problem walkthroughs, and get web-searched answers for topics beyond the knowledge base. The agent uses RAG to ground responses in a curated 12-document knowledge base and maintains multi-turn conversation memory per session.

**Knowledge base:** 12 documents covering: Newton's Laws, Kinematics & Projectile Motion, Work/Energy/Power, Thermodynamics, Electric Fields & Coulomb's Law, Magnetism & Electromagnetic Induction, Waves & Sound, Optics, Modern Physics (Quantum & Relativity), Circular Motion & Gravitation, Simple Harmonic Motion, and Fluid Mechanics.

**Tool used:** Web Search (DuckDuckGo via `ddgs`). Useful when students ask about recent physics research, problems not in the KB, or want supplementary worked examples from external sources.

**RAGAS baseline scores:**
- Faithfulness: *(run Part 6 to populate)*
- Answer Relevance: *(run Part 6 to populate)*
- Context Precision: *(run Part 6 to populate)*

**Test results:** *(run Part 5 to populate)* / 10 tests passed. Red-team: *(run Part 5 to populate)* / 2 passed.

**One thing I would improve with more time:** Add a hybrid BM25 + vector search to improve context precision for formula-heavy queries, and load real physics textbook chapters (e.g., Halliday/Resnick) as PDF documents instead of hand-written summaries.

**Most surprising thing I learned building this:** How much the router node quality affects end-to-end performance — a poorly tuned router sends too many questions to web search, bypassing the rich local knowledge base entirely.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*